In [ ]:
!pip install sentence-transformers faiss-cpu pandas gradio

In [ ]:
!pip install sentence-transformers faiss-cpu pandas gradio

In [ ]:
import pandas as pd

df = pd.read_csv("mental_health_dataset.csv")

# VERY IMPORTANT
df.columns = df.columns.str.strip()

print("Dataset loaded:", len(df))
print(df.columns)   # check columns

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(df['Prompt'].astype(str).tolist())

print("Embeddings ready")

In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("FAISS ready")

In [ ]:
def chatbot(query):

    query_vector = model.encode([query])

    distances, indices = index.search(query_vector, k=3)

    results = df.iloc[indices[0]]

    # 🚨 Crisis detection
    crisis_words = ["hurt myself", "kill myself", "suicide", "end my life"]

    for word in crisis_words:
        if word in query.lower():
            return "⚠️ I’m really sorry you're feeling this way. Please talk to someone you trust or contact a helpline immediately."

    # 🚨 Severity check
    if "high" in results['Severity'].astype(str).values:
        return results[results['Severity'] == "high"]['Solution'].iloc[0]

    # Normal response
    return results['Solution'].iloc[0]

In [26]:
import gradio as gr

def chatbot_ui(user_input):
    return chatbot(user_input)

interface = gr.Interface(
    fn=chatbot_ui,

    inputs=gr.Textbox(
        lines=5,
        placeholder="💬 Tell me what's on your mind...",
        label="Your Message"
    ),

    outputs=gr.Textbox(
        lines=8,
        label="🧠 Chatbot Response"
    ),

    title="🧠 AI Mental Health Chatbot",

    description="AI chatbot using RAG + FAISS to provide safe and supportive responses.",

    theme="soft",
    allow_flagging="never"
)

interface.launch(debug=True)

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://6c6e47f6f29e8127f7.gradio.live
